In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('career_prediction_dataset.csv')

: 

In [ ]:
# --- Basic Inspection ---
print("First 5 rows of the dataset:")
df.head()

In [ ]:
print("\nDataset Info:")
df.info()

In [ ]:
# --- Check Class Distribution ---
# Let's see if we have 100 samples for each career
plt.figure(figsize=(10, 8))
sns.countplot(y='career', data=df, order=df['career'].value_counts().index)
plt.title('Number of Samples per Career')
plt.xlabel('Count')
plt.ylabel('Career')
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualize a Key Pattern ---
# Let's check the average 'interest_programming' score for each career.
# We expect roles like Software Engineer and Game Developer to be high.
plt.figure(figsize=(10, 8))
# Group by career and get the mean, then sort
career_programming_interest = df.groupby('career')['interest_programming'].mean().sort_values(ascending=False)
sns.barplot(x=career_programming_interest.values, y=career_programming_interest.index, palette='viridis')
plt.title('Average Interest in Programming by Career')
plt.xlabel('Average Interest Score (1-5)')
plt.ylabel('Career')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# The dataframe 'df' is already loaded from the previous step

# --- 1. Separate Features (X) and Labels (y) ---
X = df.drop('career', axis=1) # All columns except the 'career' column
y = df['career']             # Only the 'career' column

print("Features (X) shape:", X.shape)
print("Labels (y) shape:", y.shape)

In [ ]:
# --- 2. Encode Text Labels into Numbers ---
# A neural network can't understand "Software Engineer", so we convert it to a number like 22.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# You can see the mapping like this:
# print("\nLabel mapping:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name}: {i}")

In [ ]:
# --- 3. Scale Numerical Features ---
# This standardizes the features (e.g., scales them to have mean=0, stddev=1).
# It's crucial for helping the neural network train efficiently.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# --- 4. Split Data into Training and Testing Sets ---
# We'll use 80% of the data to train the model, and 20% to test its performance.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,      # 20% of data will be for testing
    random_state=42,    # Ensures the split is the same every time you run
    stratify=y_encoded  # Ensures the class distribution is the same in train and test sets
)

print("\nTraining data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib

print("--- Starting Model Retraining ---")

# --- 1. Load the FULL, Expanded Dataset ---
try:
    df = pd.read_csv('career_prediction_dataset.csv')
    print(f"Dataset loaded successfully with {len(df)} rows.")
except FileNotFoundError:
    print("Error: 'career_prediction_dataset.csv' not found. Please ensure the file is in the correct directory.")
    exit()

# --- 2. Automatically Determine the Number of Classes ---
num_classes = df['career'].nunique()
print(f"Found {num_classes} unique career classes in the dataset.")


# --- 3. Preprocessing ---
# Separate Features (X) and Labels (y)
X = df.drop('career', axis=1)
y = df['career']

# Encode Text Labels into Numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Scale Numerical Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded # Crucial for imbalanced datasets
)
print("Data preprocessing complete.")
print(f"Training data shape: {X_train.shape}")
print(f"Number of output classes: {num_classes}")


# --- 4. Define the Upgraded Model Architecture ---
model = Sequential([
    # Input Layer
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3), # ADDED: Dropout layer to prevent overfitting

    # Hidden Layer 1
    Dense(128, activation='relu'),
    Dropout(0.3), # ADDED: Another Dropout layer

    # Hidden Layer 2
    Dense(64, activation='relu'),

    # Output Layer: Automatically uses the correct number of neurons
    Dense(num_classes, activation='softmax')
])

# --- 5. Compile the Model ---
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Model Summary ---")
model.summary()


# --- 6. Train the Model ---
print("\n--- Starting Model Training ---")
history = model.fit(
    X_train,
    y_train,
    epochs=60,  # Increased epochs slightly for the larger dataset
    batch_size=32,
    validation_split=0.2
)
print("--- Model Training Finished ---")


# --- 7. Evaluate the New Model ---
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n--- Evaluation on Test Data ---")
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test Loss: {loss:.4f}")


# --- 8. Save the Final, Upgraded Assets ---
print("\n--- Saving final model and transformers ---")
model.save('career_prediction_model.h5')
joblib.dump(scaler, 'scaler.gz')
joblib.dump(label_encoder, 'label_encoder.gz')
print("All files saved successfully. Your application is ready to be used with the new model!")

In [ ]:
# --- Evaluate the model on the test set ---
loss, accuracy = model.evaluate(X_test, y_test)

print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
print(f"Test Loss: {loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# --- Plot Training & Validation Accuracy ---
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# --- Plot Training & Validation Loss ---
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import joblib
from tensorflow.keras.models import load_model

# --- 1. Save the Trained Model ---
# This saves the model's architecture, weights, and training configuration.
model.save('career_prediction_model.h5')

# --- 2. Save the Scaler ---
# We need this to apply the exact same scaling to new user input.
joblib.dump(scaler, 'scaler.gz')

# --- 3. Save the Label Encoder ---
# We need this to convert the model's numerical output back into a career name.
joblib.dump(label_encoder, 'label_encoder.gz')

print("Model, scaler, and label encoder have been saved successfully!")